# Digital Twin — Event Risk Predictor

This notebook is **separate from `solution.ipynb`** (which is your official
demand-forecasting submission). This one builds the model behind your three
pitch features:

1. **Event Digital Twin** — predict `risk_level` for a hypothetical event
2. **Resource Allocation** — recommend officers/barricades/vehicles from crowd size
3. **Diversion Simulator** — compare delay estimates across scenarios

Data source: `data/processed_events.csv` (built by `prepare_dataset.py`)


## 1. Load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

df = pd.read_csv(DATA_DIR / "processed_events.csv")
print(df.shape)
df.head()

## 2. Quick sanity check

Confirm the columns we need exist and look at the target distribution
(`risk_level` is imbalanced — Severe is rare — so we'll need to handle
that during training, not just chase raw accuracy).

In [ ]:
print(df["risk_level"].value_counts())
print()
print(df["risk_level"].value_counts(normalize=True).round(3))

In [ ]:
df["risk_level"].value_counts().plot(kind="bar", color=["#1D9E75","#EF9F27","#E24B4A"])
plt.title("Risk level distribution")
plt.ylabel("count")
plt.show()

## 3. Feature selection

We predict `risk_level` using only information that would be available
**before** an event happens — this matters because the whole point of the
Digital Twin is "what if" simulation before approval, not after-the-fact
analysis. So we exclude anything that's only known after resolution
(e.g. `resolution_minutes`, `status`).

In [ ]:
feature_cols = [
    "event_cause",      # type of event (rally, accident, procession, etc.)
    "crowd_proxy",       # Low / Medium / High (derived earlier)
    "zone",               # broad location
    "hour",                 # time of day
    "day_of_week",
    "is_weekend",
]
target_col = "risk_level"

model_df = df[feature_cols + [target_col]].copy()

# hour has a few missing values (unparseable timestamps) - drop those rows
model_df = model_df.dropna(subset=["hour"])
print("Rows available for training:", len(model_df))
model_df.head()

## 4. Encode categorical features

In [ ]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = ["event_cause", "crowd_proxy", "zone", "day_of_week"]
encoders = {}

encoded_df = model_df.copy()
for col in categorical_cols:
    le = LabelEncoder()
    encoded_df[col] = le.fit_transform(encoded_df[col].astype(str))
    encoders[col] = le  # keep these - you need them later to decode user input in Streamlit

encoded_df["is_weekend"] = encoded_df["is_weekend"].astype(int)
encoded_df.head()

## 5. Train / test split

In [ ]:
from sklearn.model_selection import train_test_split

X = encoded_df[feature_cols]
y = encoded_df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)

## 6. Train a Random Forest classifier

`class_weight="balanced"` matters here — without it, the model will mostly
learn to predict "Moderate" since it's the majority class, and Severe
(the most important one for police!) will barely get learned at all.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
clf.fit(X_train, y_train)
print("Trained.")

## 7. Evaluate

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=clf.classes_)
disp.plot(cmap="Blues")
plt.title("Confusion matrix - risk_level")
plt.show()

**How to read this:** look at `recall` for the `Severe` row in the report
above. That's the metric that matters most for a safety tool — it tells you
what fraction of genuinely severe events your model successfully flags as
severe. A high accuracy number can hide a model that misses every Severe
case, so don't just report accuracy in your demo.

## 8. Feature importance — what's actually driving predictions?

In [ ]:
importances = pd.Series(clf.feature_importances_, index=feature_cols).sort_values()
importances.plot(kind="barh", color="#7F77DD")
plt.title("Feature importance")
plt.xlabel("importance")
plt.show()

## 9. The Digital Twin function

This is the function your Streamlit app will call. Give it event details,
get back a risk prediction — this is feature #1 from your pitch.

In [ ]:
def predict_event_risk(event_cause, crowd_level, zone, hour, day_of_week, is_weekend):
    """
    event_cause: one of df['event_cause'].unique()  e.g. 'public_event', 'procession'
    crowd_level: 'Low' / 'Medium' / 'High'
    zone: one of df['zone'].unique()
    hour: int 0-23
    day_of_week: 'Monday'...'Sunday'
    is_weekend: bool
    """
    row = pd.DataFrame([{
        "event_cause": encoders["event_cause"].transform([event_cause])[0]
            if event_cause in encoders["event_cause"].classes_ else 0,
        "crowd_proxy": encoders["crowd_proxy"].transform([crowd_level])[0]
            if crowd_level in encoders["crowd_proxy"].classes_ else 0,
        "zone": encoders["zone"].transform([zone])[0]
            if zone in encoders["zone"].classes_ else 0,
        "hour": hour,
        "day_of_week": encoders["day_of_week"].transform([day_of_week])[0]
            if day_of_week in encoders["day_of_week"].classes_ else 0,
        "is_weekend": int(is_weekend),
    }])[feature_cols]

    pred = clf.predict(row)[0]
    proba = clf.predict_proba(row)[0]
    proba_dict = dict(zip(clf.classes_, proba.round(3)))

    return {
        "risk_level": pred,
        "confidence": proba_dict,
        "requires_road_closure_likely": pred == "Severe",
    }

# quick test
predict_event_risk(
    event_cause="public_event",
    crowd_level="High",
    zone="Central Zone 2",
    hour=18,
    day_of_week="Saturday",
    is_weekend=True,
)

## 10. Resource Allocation Intelligence (Feature #2)

Simple rule-based formula on top of the crowd estimate — this is the
"actionable insight" layer. No ML needed here, just sensible ratios you
can defend to judges.

In [ ]:
def recommend_resources(crowd_size, risk_level):
    """
    crowd_size: estimated number of people (int)
    risk_level: 'Minor' / 'Moderate' / 'Severe' (from predict_event_risk)
    """
    officers = max(2, crowd_size // 500)
    barricades = max(2, crowd_size // 1500)
    patrol_vehicles = max(1, crowd_size // 4000)

    # scale up for higher risk events - same crowd size but higher risk
    # still needs a stronger response
    risk_multiplier = {"Minor": 1.0, "Moderate": 1.2, "Severe": 1.5}.get(risk_level, 1.0)

    return {
        "traffic_officers": int(officers * risk_multiplier),
        "barricades": int(barricades * risk_multiplier),
        "patrol_vehicles": int(patrol_vehicles * risk_multiplier),
    }

# quick test
recommend_resources(crowd_size=20000, risk_level="Severe")

## 11. Dynamic Diversion Simulator (Feature #3)

This is a simplified scenario comparison — for the hackathon demo, base
delay estimates on `risk_level` and `crowd_proxy` rather than a full
routing engine (that's out of scope for the time you have).

You can make this more sophisticated later if you have time, e.g. by
pulling real road network distances. For now this gives judges a
believable, explainable comparison.

In [ ]:
def simulate_diversions(risk_level, crowd_level):
    """Returns 3 scenarios with estimated delay in minutes."""
    base_delay = {"Minor": 15, "Moderate": 30, "Severe": 50}.get(risk_level, 20)
    crowd_addon = {"Low": 0, "Medium": 10, "High": 20}.get(crowd_level, 0)

    no_diversion = base_delay + crowd_addon
    route_b = round((no_diversion) * 0.55)
    early_closure = round((no_diversion) * 0.35)

    scenarios = {
        "No diversion":      no_diversion,
        "Route B diversion": route_b,
        "Early road closure": early_closure,
    }
    best = min(scenarios, key=scenarios.get)
    return {"scenarios": scenarios, "recommended": best}

# quick test
simulate_diversions(risk_level="Severe", crowd_level="High")

## 12. End-to-end test — the full Digital Twin pipeline

This is what your Streamlit form will run when police submit an event.

In [ ]:
def run_digital_twin(event_cause, crowd_level, zone, hour, day_of_week, is_weekend):
    crowd_estimate_map = {"Low": 500, "Medium": 5000, "High": 20000}
    crowd_size = crowd_estimate_map[crowd_level]

    risk_result = predict_event_risk(event_cause, crowd_level, zone, hour, day_of_week, is_weekend)
    resources = recommend_resources(crowd_size, risk_result["risk_level"])
    diversions = simulate_diversions(risk_result["risk_level"], crowd_level)

    return {
        "risk_level": risk_result["risk_level"],
        "confidence": risk_result["confidence"],
        "resources_needed": resources,
        "diversion_scenarios": diversions["scenarios"],
        "recommended_diversion": diversions["recommended"],
    }

# Example: a political rally
import json
result = run_digital_twin(
    event_cause="public_event",
    crowd_level="High",
    zone="Central Zone 2",
    hour=18,
    day_of_week="Saturday",
    is_weekend=True,
)
print(json.dumps(result, indent=2))

## 13. Save the trained model

Save this so your Streamlit app can load it directly without retraining
every time.

In [ ]:
import joblib

MODEL_DIR = PROJECT_ROOT / "src"
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(clf, MODEL_DIR / "risk_model.pkl")
joblib.dump(encoders, MODEL_DIR / "risk_encoders.pkl")
print("Saved model and encoders to", MODEL_DIR)

## Next steps

- [ ] Build the Streamlit app (`app.py`) that calls `run_digital_twin()`
- [ ] Add a dropdown UI for event_cause, zone, crowd_level, hour, day
- [ ] Display risk_level with color coding (green/orange/red)
- [ ] Show resource recommendations as metric cards
- [ ] Show the 3 diversion scenarios as a bar chart, highlight the best one
- [ ] (Optional, time permitting) Replace the rule-based diversion delays
      with something derived from real `resolution_minutes` data once you
      have more than 69 rows of it
